# 

In [1]:
import numpy as np
import inspect
import importlib
import Utils
importlib.reload(Utils)
from rasterstats import zonal_stats
import Constants
importlib.reload(Constants)
import ConstantObjects
importlib.reload(ConstantObjects)
import matplotlib.pyplot as plt
import os
import rasterio
from rasterio.warp import reproject, Resampling

print("Done with cell!")



[Line 13] n_cols in ConstantObjects: 21
[Line 15] n_cols: 21, n_rows: 18
[Line 26] n_cols in ConstantObjects: 21
[Line 28] gdf_tree_circles.shape: (378, 1)
x0 =  4246989.3147608945
y0 =  547032.0335418215
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (4.907739, 38.150835)
x0 =  4246989.3147608945
y0 =  547032.0335418215
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (4.907722, 38.151140)
x0 =  4246866.863321021
y0 =  546964.9960692101
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (4.907036, 38.149684)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  4246989.3147608945
y0 =  546674.5010509877
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (4.908725, 38.157014)
x0 =  4246989.3147608945
y0 =  547032.0335418215
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (4.907594, 38.151076)
x0 =  4247044.974506292
y0 =  547177.2816059613
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (4.908

In [2]:
#dates = Utils.generate_date_range("2025-01-25", "2025-01-26",  "%Y-%m-%d", 5)

#NDMI = Normalized Difference Moisture Index
# Output lists
ndmi_values = []
valid_dates = []




In [ ]:
# Re-run the previous code after reset
#port Isabel dates
#veracruz good dates
#2024-01-13
#2025-01-12
#dates = Utils.generate_date_range("2025-01-02", "2025-06-10",  "%Y-%m-%d", 5)
#dates = Utils.generate_date_range("2025-01-02", "2025-01-16",  "%Y-%m-%d", 5)
#ates = Utils.generate_date_range("2024-01-13", "2025-09-13",  "%Y-%m-%d", 5)
dates = Utils.generate_date_range("2022-01-03", "2025-09-13",  "%Y-%m-%d", 5)
#dates = Utils.generate_date_range("2024-05-02", "2025-09-13",  "%Y-%m-%d", 5)
#dates = Utils.generate_date_range("2022-01-03", "2022-01-04",  "%Y-%m-%d", 5)
#dates = Utils.generate_date_range("2025-02-11", "2025-02-12",  "%Y-%m-%d", 5)
#Borana University dates:
#jan 2020 to december 2022 -- horn of africa drought period
dates = Utils.generate_date_range("2021-09-30", "2022-06-30",  "%Y-%m-%d", 5) #Renaud wanted to start 2021-10-01. But need to see to it that it goes through 2022-01-03, because those dates work for sentinel2, coincidentally enough.
dates = Utils.generate_date_range("2016-09-29", "2022-06-30",  "%Y-%m-%d", 5) # 2015-01-01 does not have data, 2015-01-02 . 2015-01-05 and 2015-01-06: found mgrs, not others. 2020-09-30 works
dates = Utils.generate_date_range("2017-12-28", "2022-06-30",  "%Y-%m-%d", 5) # restart after interruption
#2019-09-30, no data. 2019-09-29 works. 2018-09-29, 2017-09-29, 2016-09-29 also works.
#2015-09-29, 2015-09-28, 2015-09-26, 2015-09-26 fails. Could be because sentinel2 launched in june 2015
print("dates = ",dates)

#NDMI = Normalized Difference Moisture Index
# Output lists
ndmi_values = []
valid_dates = []



# Prepare folder
#os.makedirs("s2_point_series", exist_ok=True)
output_tif_cumulative = "ndmi_cumulative.tif"
output_tif = ""
gndvi_tif = ""
rgb_tif = ""

# Sample NDMI at the given point
for date in dates:
    print("processing date ",date)
    # Download images corresponding to bands 2,3,4,8,11.
    b02_path = Utils.download_band_dynamic(date, "B02",Constants.lat0,Constants.lon0)  # Blue
    b03_path = Utils.download_band_dynamic(date, "B03",Constants.lat0,Constants.lon0)  # Green
    b04_path = Utils.download_band_dynamic(date, "B04",Constants.lat0,Constants.lon0)  # Red
    b08_path = Utils.download_band_dynamic(date, "B08",Constants.lat0,Constants.lon0)
    print(f"[Cell Line {inspect.currentframe().f_lineno}] ...  ")
    b11_path = Utils.download_band_dynamic(date, "B11",Constants.lat0,Constants.lon0)
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ...  ")
    #b04_path = Utils.download_band_dynamic(date, "B04",Constants.lat0,Constants.lon0)
    
    print("set b02_path ",  b02_path)
    print("set b03_path ",  b03_path)
    print("set b04_path ",  b04_path)
    print("set b08_path ",  b08_path)
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ...  ")
    print("set b11_path ",  b11_path)


    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... About to try ")
    try:
        with rasterio.open(b08_path) as src_b08, rasterio.open(b11_path) as src_b11, rasterio.open(b03_path) as src_b03 , rasterio.open(b04_path) as src_b04 : # , rasterio.open(b02_path) as src_b02, rasterio.open(b03_path) as src_b03, rasterio.open(b04_path) as src_b04:
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            b11_data = src_b11.read(1)
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            b03_data = src_b03.read(1)
            b04_data = src_b04.read(1)
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            resampled_b11 = np.empty((10980, 10980), dtype="float32")
            # Create an ndmi tif for this date
            output_tif = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "ndmi","tif")
            os.path.exists(output_tif) or (Utils.write_ndmi_geotiff(  b08_path, b11_path, output_tif), print("✅ NDMI GeoTIFF in UTM saved to:", output_tif))
            out_dtype = "uint8"
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            rgb_tif    = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb."+out_dtype,"tif")
            os.path.exists(rgb_tif) or (Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path, rgb_tif, out_dtype), print("✅ RGB GeoTIFF in UTM saved to:", rgb_tif))
            out_dtype = "float32"
            rgb_tif    = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb."+out_dtype,"tif")
            os.path.exists(rgb_tif) or (Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path, rgb_tif, out_dtype), print("✅ RGB GeoTIFF in UTM saved to:", rgb_tif) )    
            out_dtype = "float32"
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            gndvi_tif = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "gndvi","tif")
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")
            os.path.exists(gndvi_tif) or (Utils.write_gndvi_geotiff(  b08_path, b03_path, gndvi_tif), print("✅ GNDVI GeoTIFF in UTM saved to:", gndvi_tif))
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")


            #ndmi_3857 = Utils.reproject_UTM_to_3857(output_tif, "ndmi_3857.tif")
            
    
                
    except Exception as e:
        print(f"Skipping {date} due to error: {e}")


print(f"Cell [Line {inspect.currentframe().f_lineno}] ... Done with cell! ")


dates =  ['2017-12-28', '2018-01-02', '2018-01-07', '2018-01-12', '2018-01-17', '2018-01-22', '2018-01-27', '2018-02-01', '2018-02-06', '2018-02-11', '2018-02-16', '2018-02-21', '2018-02-26', '2018-03-03', '2018-03-08', '2018-03-13', '2018-03-18', '2018-03-23', '2018-03-28', '2018-04-02', '2018-04-07', '2018-04-12', '2018-04-17', '2018-04-22', '2018-04-27', '2018-05-02', '2018-05-07', '2018-05-12', '2018-05-17', '2018-05-22', '2018-05-27', '2018-06-01', '2018-06-06', '2018-06-11', '2018-06-16', '2018-06-21', '2018-06-26', '2018-07-01', '2018-07-06', '2018-07-11', '2018-07-16', '2018-07-21', '2018-07-26', '2018-07-31', '2018-08-05', '2018-08-10', '2018-08-15', '2018-08-20', '2018-08-25', '2018-08-30', '2018-09-04', '2018-09-09', '2018-09-14', '2018-09-19', '2018-09-24', '2018-09-29', '2018-10-04', '2018-10-09', '2018-10-14', '2018-10-19', '2018-10-24', '2018-10-29', '2018-11-03', '2018-11-08', '2018-11-13', '2018-11-18', '2018-11-23', '2018-11-28', '2018-12-03', '2018-12-08', '2018-12-1

ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed
ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed
ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed
ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed
ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed
ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed


Cell [Line 67] ... 
[Line 124] ... Mapped latitude  4.908058823095451 , longitude  38.15135412942004  to mgrs_tile =  37NCF
band = > ndmi <
extension = > tif <
Cell [Line 74] ... 
[Line 124] ... Mapped latitude  4.908058823095451 , longitude  38.15135412942004  to mgrs_tile =  37NCF
band = > rgb.uint8 <
extension = > tif <
[Line 784] ...       


ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed
ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed
ERROR 1: Stream too short

ERROR 1: opj_get_decoded_tile() failed


[Line 812] ...       
[Line 833] ...       
Skipping 2017-12-28 due to error: No valid pixels found to compute stretch.
processing date  2018-01-02
[Line 124] ... Mapped latitude  4.908058823095451 , longitude  38.15135412942004  to mgrs_tile =  37NCF
[Line 124] ... Mapped latitude  4.908058823095451 , longitude  38.15135412942004  to mgrs_tile =  37NCF
[Line 124] ... Mapped latitude  4.908058823095451 , longitude  38.15135412942004  to mgrs_tile =  37NCF
[Line 124] ... Mapped latitude  4.908058823095451 , longitude  38.15135412942004  to mgrs_tile =  37NCF
[Cell Line 44] ...  
[Line 124] ... Mapped latitude  4.908058823095451 , longitude  38.15135412942004  to mgrs_tile =  37NCF
Cell [Line 46] ...  
b02_path  s2_point_series/37NCF_2018-01-02_B02.jp2
b03_path  s2_point_series/37NCF_2018-01-02_B03.jp2
b04_path  s2_point_series/37NCF_2018-01-02_B04.jp2
b08_path  s2_point_series/37NCF_2018-01-02_B08.jp2
Cell [Line 53] ...  
b11_path  s2_point_series/37NCF_2018-01-02_B11.jp2
Cell [Line 57]

In [ ]:
import leafmap
#importlib.reload(ConstantObjects)
print(f"[Line {inspect.currentframe().f_lineno}] ... lat0 = ",Constants.lat0)
print(f"[Line {inspect.currentframe().f_lineno}] ... lon0 = ",Constants.lon0)

m2 = leafmap.Map(basemap="Esri.WorldImagery")
#m2 = leafmap.Map(center=(Constants.lat0,Constants.lon0), zoom=15)
m2.add_raster(output_tif, layer_name="NDMI", opacity=0.9, nodata=float("nan"))

m2.set_center(Constants.lon0, Constants.lat0,  zoom=15)

            
m2.add_gdf(ConstantObjects.gdf_box_terreno_casa, layer_name="Plot Boundary")
m2.add_gdf(ConstantObjects.gdf_box_plots_4_5, layer_name="Plots 4,5")
m2.add_gdf(ConstantObjects.gdf_box_pueblo, layer_name="pueblo")
m2.add_gdf(ConstantObjects.gdf_box_river, layer_name="river")

#show the entire ndmi tile
#m2.add_raster(output_tif_cumulative, layer_name="NDMI", colormap="BrBG", nodata=np.nan, opacity = .55)
#print(f"[Line {inspect.currentframe().f_lineno}] ... b04_path = ",b04_path)
#m2.add_raster(b04_path, layer_name="Red", colormap="BrBG", nodata=np.nan, opacity = .9)



#ndmi_3857 = reproject_to_3857("ndmi_14QQG.tif", "ndmi_14QQG_3857.tif")
#m2.add_raster(output_tif, layer_name="NDMI", opacity=0.9)

#m2.add_raster(ndmi_3857, layer_name="NDMI", opacity=0.9, nodata=float("nan"))


m2